In [0]:
import sys
import os
sys.path.insert(0, os.getcwd())

import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorRefreshInfoState
from utils import load_project, table

cfg = load_project()
dbutils.widgets.text("catalog", "stock_demo")
catalog = dbutils.widgets.get("catalog")
print(f"Refreshing monitor for catalog: {catalog}")

table_name = table(cfg, "inference_log", catalog)
w = WorkspaceClient()

print(f"Triggering refresh for monitor on {table_name}")
refresh = w.quality_monitors.run_refresh(table_name=table_name)
refresh_id = refresh.refresh_id
print(f"Refresh started: {refresh_id}")

TERMINAL_STATES = (MonitorRefreshInfoState.SUCCESS, MonitorRefreshInfoState.FAILED,
                   MonitorRefreshInfoState.CANCELED)
max_wait_seconds, waited, status = 3600, 0, None
while waited < max_wait_seconds:
    status = w.quality_monitors.get_refresh(table_name=table_name, refresh_id=refresh_id)
    print(f"Refresh state: {status.state}")
    if status.state in TERMINAL_STATES:
        break
    time.sleep(15)
    waited += 15

if status is None or status.state != MonitorRefreshInfoState.SUCCESS:
    raise RuntimeError(f"Monitor refresh did not succeed: {status.state if status else 'timed out'}")
print("Monitor refresh completed successfully.")